# Hyperparameter Tuning for Dataset: MC2
Thử nghiệm và tìm kiếm bộ siêu tham số tốt nhất cho các mô hình máy học trên tập dữ liệu `MC2.arff`.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import sys
import os
# Đảm bảo có thể import module từ thư mục src
sys.path.append(os.path.abspath('..'))

from src.data_loader import load_arff_dataset
from src.models import SoftwareDefectPredictionEngine

# 1. Tải dữ liệu
dataset_path = '../datasets/MC2.arff'
df, target_col = load_arff_dataset(dataset_path)
df = df.dropna()

X = df.drop(columns=[target_col]).values
y = df[target_col].map({'Defective': 1, 'Clean': 0}).values

print(f"Dataset Shape: {X.shape}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 2. Chia tập huấn luyện / kiểm thử
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

engine = SoftwareDefectPredictionEngine(random_state=42)

best_params_report = {}

# 3. Tiến hành tìm kiếm tham số tối ưu (Hyperparameter Tuning)
for model_name, model in engine.classifiers.items():
    print(f"\n--- Tuning {model_name} ---")
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])
    
    param_grid = engine.get_param_grid(model_name)
    
    if param_grid:
        search = RandomizedSearchCV(
            pipeline, 
            param_distributions=param_grid, 
            n_iter=15, 
            cv=3, 
            scoring='f1_weighted', 
            random_state=42, 
            n_jobs=-1
        )
        search.fit(X_train, y_train)
        
        # Bỏ tiền tố 'model__' khi in ra cho dễ đọc
        best_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
        print(f"Best Params: {best_params}")
        print(f"Best CV Score (F1): {search.best_score_:.4f}")
        best_params_report[model_name] = best_params
    else:
        print("No parameters to tune.")
        best_params_report[model_name] = "Default"


In [ ]:
# 4. Hiển thị tổng hợp cấu hình tốt nhất
print("\n\n==== BÁO CÁO CẤU HÌNH TỐT NHẤT CHO MC2 ====")
for model, params in best_params_report.items():
    print(f"{model}: {params}")
